# Overview

This figure shows the effect of the shuffling and swapping on the input
topography :scope: paper :figure: 5

## Data source

# Setup

``` python
import sqlite3
import numpy as np
from landlab_torch_tools import HorizontalSwap, GridShuffle
from neural_spd.config import PROJECT_ROOT
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from neural_spd import plot_styles
import torch
plot_styles.apply()
db_path = PROJECT_ROOT / "model_runs.db"
data_dir = PROJECT_ROOT / "data/0/elevation"
```

# Load and shuffle data

## Select characteristic dataset

``` python
connection = sqlite3.connect(db_path)
cursor = connection.cursor()
cursor.execute("SELECT model_run_id, \"model_param.diffuser.D\" / \"model_param.streampower.k\" FROM model_run_params")
results = cursor.fetchall()
# sort by second row
results.sort(key=lambda x: x[1])
# select 10th percentile run
tenth_percentile_index = int(len(results) * 0.1)
selected_run_id = results[tenth_percentile_index][0]
elev = torch.tensor(np.load(data_dir / f"{selected_run_id}.npy"))
```

## Apply shuffling and swapping

``` python
hswap = HorizontalSwap()
shuff = GridShuffle()
elev_hswap = hswap(elev)
elev_shuff = shuff(elev)
```

# Plotting function

``` python
_TITLES = ["Original", "Horizontal Swap", "Grid Shuffle"]
_ELEVS  = [None, None, None]   # filled at call time to avoid stale refs

def plot_maps(axs, elev, elev_hswap, elev_shuff):
    vmin = elev.min()
    vmax = elev.max()
    for ax, data, title in zip(axs, [elev, elev_hswap, elev_shuff], _TITLES):
        im = ax.imshow(data, cmap="terrain", vmin=vmin, vmax=vmax,
                       interpolation="nearest")
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
    # shared colorbar on the rightmost panel
    divider = make_axes_locatable(axs[-1])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cb = plt.colorbar(im, cax=cax)
    cb.set_label("Elevation (m)", fontsize=plt.rcParams["axes.labelsize"])
    cb.ax.tick_params(labelsize=plt.rcParams["xtick.labelsize"])
```

# Generate plots

``` python
plot_styles.double_column()
fig, axs = plt.subplots(1, 3, figsize=(6.75, 2.4))
plot_maps(axs, elev, elev_hswap, elev_shuff)
plot_styles.save_figure(fig, "shuffle_viz", PROJECT_ROOT  / "paper/figs")
fig.show()
```